<a href="https://colab.research.google.com/github/asmaa-2003/LSTM-Anomaly--detection/blob/main/metropt3_predictive_maintenance_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔧 MetroPT3 — Predictive Maintenance (Version Corrigée)
Détection d'anomalies et prédiction de pannes sur compresseur d'air (métro de Porto).

## ⚙️ 1. Imports & Config

In [3]:
import warnings, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xgboost as xgb
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score, roc_auc_score

DATA_PATH = "MetroPT3.csv"

WINDOW_HOURS  = 6
STEP          = '5min'
WARNING_HOURS = 48   # ✅ زدناها باش نلقاو positives
RANDOM_STATE  = 42

SENSORS = [
    'TP2', 'TP3', 'H1', 'DV_pressure',
    'Reservoirs', 'Oil_temperature', 'Motor_current'
]

# ✅ فقط failure الموجود داخل البيانات
FAILURES = [
    ('2020-04-18', '2020-04-19'),
]

In [6]:
df = pd.read_csv(DATA_PATH)

# ✅ تحويل timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'])

df = df.sort_values('timestamp').reset_index(drop=True)
df = df.drop_duplicates('timestamp')
df = df.ffill().bfill()

print(df['timestamp'].min(), df['timestamp'].max())

2020-02-01 00:00:00 2020-02-02 05:13:28


In [7]:
df['label'] = 0

for start, end in FAILURES:
    t0 = pd.Timestamp(start)
    t1 = pd.Timestamp(end)

    warning_start = t0 - pd.Timedelta(hours=WARNING_HOURS)

    # warning
    df.loc[(df['timestamp'] >= warning_start) & (df['timestamp'] < t0), 'label'] = 1

    # failure
    df.loc[(df['timestamp'] >= t0) & (df['timestamp'] <= t1), 'label'] = 2

print(df['label'].value_counts())

label
0    7381
Name: count, dtype: int64


In [8]:
def slope(series):
    if len(series) < 2:
        return 0.0
    x = np.arange(len(series))
    return np.polyfit(x, series, 1)[0]

df = df.set_index('timestamp')

times = pd.date_range(
    df.index.min() + pd.Timedelta(hours=WINDOW_HOURS),
    df.index.max(),
    freq=STEP
)

X_list, y_list = [], []

for t in times:
    window = df.loc[t - pd.Timedelta(hours=WINDOW_HOURS): t]

    if len(window) < 50:
        continue

    # ✅ لا نحذف كل windows (مهم جدًا)
    if window['label'].mean() > 0.8:
        continue

    row = {}

    for col in SENSORS:
        s = window[col].values

        row[f'{col}_mean']  = s.mean()
        row[f'{col}_std']   = s.std()
        row[f'{col}_max']   = s.max()
        row[f'{col}_min']   = s.min()
        row[f'{col}_trend'] = slope(s)

    row['hour'] = t.hour

    # 🎯 target
    future = df.loc[t : t + pd.Timedelta(hours=WARNING_HOURS)]
    y_list.append(int((future['label'] == 2).any()))

    X_list.append(row)

X = pd.DataFrame(X_list).fillna(0)
y = np.array(y_list)

print("Total positives:", (y == 1).sum())

Total positives: 0


In [9]:
split = int(len(X) * 0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y[:split], y[split:]

In [10]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

In [11]:
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()

ratio = n_neg / n_pos if n_pos > 0 else 1

print("Neg:", n_neg, "| Pos:", n_pos)

Neg: 194 | Pos: 0


In [12]:
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(X_train_sc, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=-1, num_parallel_tree=None, ...)

In [13]:
y_proba = model.predict_proba(X_test_sc)[:,1]

threshold = 0.3
y_pred = (y_proba > threshold).astype(int)

print(classification_report(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        49

    accuracy                           1.00        49
   macro avg       1.00      1.00      1.00        49
weighted avg       1.00      1.00      1.00        49

F1: 0.0
AUC: nan


In [14]:
os.makedirs("model", exist_ok=True)

joblib.dump(model, "model/xgb_model.pkl")
joblib.dump(scaler, "model/scaler.pkl")

print("Model saved")

Model saved
